In [1]:
"""
Sentiment Analysis on Yelp Reviews

Developer: Swapnendu Banik
Version: 1.0
Description: This notebook demonstrates how to perform sentiment analysis on Yelp reviews using a pre-trained BERT model. It scrapes reviews from a Yelp page, processes them, and predicts an overall outlet rating based on the sentiment scores of individual reviews.

Models Used:
- nlptown/bert-base-multilingual-uncased-sentiment: A pre-trained BERT model from Hugging Face, fine-tuned for sentiment analysis. [BERT]

Problem Statements Solved:
- **Extracting Real-time Reviews**: Scrapes Yelp reviews for a specified restaurant.
- **Sentiment Analysis**: Uses a pre-trained model to evaluate the sentiment expressed in each review.
- **Rating Prediction**: Calculates a predicted outlet rating based on individual review sentiments.
- **Comparison**: Compares the predicted rating with the actual Yelp rating to assess the model's accuracy.
"""

"\nSentiment Analysis on Yelp Reviews\n\nDeveloper: Swapnendu Banik\nVersion: 1.0\nDescription: This notebook demonstrates how to perform sentiment analysis on Yelp reviews using a pre-trained BERT model. It scrapes reviews from a Yelp page, processes them, and predicts an overall outlet rating based on the sentiment scores of individual reviews.\n\nModels Used:\n- nlptown/bert-base-multilingual-uncased-sentiment: A pre-trained BERT model from Hugging Face, fine-tuned for sentiment analysis. [BERT]\n\nProblem Statements Solved:\n- **Extracting Real-time Reviews**: Scrapes Yelp reviews for a specified restaurant.\n- **Sentiment Analysis**: Uses a pre-trained model to evaluate the sentiment expressed in each review.\n- **Rating Prediction**: Calculates a predicted outlet rating based on individual review sentiments.\n- **Comparison**: Compares the predicted rating with the actual Yelp rating to assess the model's accuracy.\n"

# Sentiment Analysis on Yelp Reviews

## Objective
In this exercise, we will:
1. Extract real-time reviews from the [Mountain Mike's Pizza Yelp page](https://www.yelp.com/biz/mountain-mikes-pizza-daly-city-2).
2. Use a pre-trained sentiment analysis model to evaluate each review and assign it a score between **0 to 5**.
3. Calculate the average score from the reviews to determine a predicted outlet rating.
4. Compare the predicted rating with the actual Yelp rating provided for the outlet.

## Steps Overview

1. **Data Extraction**:
   - We'll scrape the review section from the Yelp page using a web scraping tool or API. Ensure you follow Yelp's terms of service for web scraping.

2. **Model Selection**:
   - Use a pre-trained sentiment analysis model capable of scoring reviews on a **0 to 4 scale**.

3. **Scoring**:
   - Each review will be processed by the model to generate a numerical score representing the sentiment of the review:
     - **1**: Very negative
     - **5**: Very positive

4. **Rating Prediction**:
   - Average the scores of all reviews to calculate the predicted outlet rating.

5. **Comparison**:
   - Compare the predicted rating with the real Yelp rating to understand the alignment between the model's sentiment analysis and Yelp's aggregated score.

## Key Notes for Beginners
- This project introduces **sentiment analysis** and demonstrates how models can provide quantitative insight into text data.
- You’ll also explore how machine learning can be used to evaluate real-world data and compare it to human-aggregated scores.
- **Caution**: While scraping, always adhere to ethical practices and platform policies to avoid violations.

By the end of this exercise, you'll have a practical understanding of applying machine learning to text data and evaluating real-world performance against pre-existing metrics.


## Important Necessary Libraries

In [2]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch
import requests
from bs4 import BeautifulSoup
import re
import textwrap

### A Different Way of Transformer Model Loading than what we saw

#### Overview
When working with transformer models, the Hugging Face library provides several ways to load models and tokenizers. Each method serves a specific purpose and use case, depending on the task at hand. Below, we discuss alternative methods for model loading, when to use them, and what each method does.

---


#### 1. **`AutoTokenizer.from_pretrained`**

##### What It Does:
- Loads a **pre-trained tokenizer** from Hugging Face's model hub or a local directory.
- A tokenizer is responsible for converting raw text into tokenized inputs (e.g., token IDs) that the model can process.
- Includes configuration, vocabulary, and tokenization rules specific to the model architecture.

##### When to Use:
- When working with pre-trained models for tasks like sentiment analysis, classification, or question-answering.
- If you want the tokenizer tailored to a specific model architecture (e.g., BERT, GPT).
- Required when preparing text inputs for any transformer model.


#### 2. **`AutoModelForSequenceClassification.from_pretrained`**

##### What It Does:
- Loads a **pre-trained transformer model** designed for sequence classification tasks.
- Includes pre-trained weights and architecture configurations.
- Supports tasks like sentiment analysis, spam detection, and other classification-based use cases.

##### When to Use:
- When working on text classification tasks (e.g., sentiment analysis, topic classification).
- If you need a model with pre-trained weights and task-specific fine-tuning capabilities.
- Paired with a tokenizer, this model can process tokenized text inputs and produce predictions.






In [3]:
tokenizer = AutoTokenizer.from_pretrained('nlptown/bert-base-multilingual-uncased-sentiment')

model = AutoModelForSequenceClassification.from_pretrained('nlptown/bert-base-multilingual-uncased-sentiment')

### Let's see the Model in action with 1 example case before jumping into a bigger application

In [4]:
reviews = ['I hated this, digusting and horrendous',
            'Not satisfied' , 
            'Its actually very good, might be 2nd best in my list' ,
            'Good place but can do better',
            'OMG!! I am going to recommend all my friends and family this place! Its simply the best place ever!!']


## We need to tokenize the text, machines dont understand words; they need to convert it to numbers before any process.

tokens = tokenizer(reviews,padding=True, truncation=True, return_tensors='pt')
# padding=True: Ensures all sequences are of the same length by adding padding.
# truncation=True: Cuts off sequences that are longer than the model's maximum input size.
# return_tensors='pt': Converts the output to PyTorch tensors.



# For a single input you will use something like
# tokens = tokenizer.encode('I hated this, digusting and horrendous', return_tensors='pt')


In [5]:
print("Unique tokens created :",len(tokens['input_ids'])) # 5 sentences -> 5 tokens
print("--------------------------------------------------------------------------------------------------------------------------")
query_index=1 ## Play around with query_index val to visualize each sentences
print(f"Token for sentence {query_index}: ",tokens['input_ids'][query_index], "Length of the token: ",len(tokens['input_ids'][query_index]) )
print("--------------------------------------------------------------------------------------------------------------------------")
print(f"Decoded version of the same sentence: ",tokenizer.decode(tokens['input_ids'][query_index]))
print("--------------------------------------------------------------------------------------------------------------------------")

Unique tokens created : 5
--------------------------------------------------------------------------------------------------------------------------
Token for sentence 1:  tensor([  101, 10497, 90606, 45220, 32245,   102,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0,     0,
            0,     0,     0,     0,     0,     0,     0,     0,     0]) Length of the token:  29
--------------------------------------------------------------------------------------------------------------------------
Decoded version of the same sentence:  [CLS] not satisfied [SEP] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD] [PAD]
--------------------------------------------------------------------------------------------------------------------------


### Let's feed them into the model and see the outputs.
Sentiments are divided into 5 classes; Very Bad, Bad, Good, Very Good, Excellent

In [6]:
results=[] ## Array to store the output rating of each token/sentence

outputs = model(**tokens)  # Note that we use **tokens to unpack the input for the model
probabilities = outputs.logits ## The logits (predictions) are stored in `logits`, 
## There are 5 outputs in the probabilities for each sentence/token; as the model has 5 output classes.
## We need to select the one with most value/probability

print("Probability of which class between [0-4] is best for the sentences: ",torch.argmax(probabilities, dim=1).tolist()) ## As prob is an list of lists, we need to mention the dim
## or else the max val within ALL is given, same as writing dim=0 (Among ALL) but dim=1 means (Among that Row)

prob_sentence_wise=torch.argmax(probabilities, dim=1).tolist()
print("--------------------------------------------------------------------------------------")
for i in range(len(prob_sentence_wise)):
    print("Rating for: ")
    print(reviews[i])
    print(prob_sentence_wise[i]+1)
    print("--------------------------------------------------------------------------------------")


Probability of which class between [0-4] is best for the sentences:  [0, 1, 3, 2, 4]
--------------------------------------------------------------------------------------
Rating for: 
I hated this, digusting and horrendous
1
--------------------------------------------------------------------------------------
Rating for: 
Not satisfied
2
--------------------------------------------------------------------------------------
Rating for: 
Its actually very good, might be 2nd best in my list
4
--------------------------------------------------------------------------------------
Rating for: 
Good place but can do better
3
--------------------------------------------------------------------------------------
Rating for: 
OMG!! I am going to recommend all my friends and family this place! Its simply the best place ever!!
5
--------------------------------------------------------------------------------------


## OK, I guess we are accustomed with the working of the model. Time for a Real World Application! 

#### Scraps reviews from yelp website for a restaurant and run our sentiment analyis model!

### Important:

When you're reading this notebook, please note that the class ID for the reviews might have changed due to dynamic class generation by the website. Websites like Yelp often use dynamically generated class names, meaning the class name used to extract review data might be different when you run the code.

**What you need to do:**

1.  **Watch the attached video**: The video demonstrates how to identify the current class name for the review section on the Yelp page.
2.  **Update the class ID**: After watching the video, find the new class ID in the HTML of the page.
3.  **Replace the old class ID**: Paste the new class ID into the regex section of the code in place of the old one.

This ensures that the scraping script continues to work as expected, even if the website’s structure changes.


<video width="1280" height="720" controls>
  <source src="tutorial-video\How to scrap reviews IRT.mp4" type="video/mp4">
  Video has been removed from the directory
</video>

In [7]:
# Step 1: Sending an HTTP request to the Yelp page to get the HTML content
r = requests.get("https://www.yelp.com/biz/mountain-mikes-pizza-daly-city-2")
# `requests.get()` sends a GET request to the specified URL and returns the response.
# The response includes the HTML content of the Yelp page.

# Step 2: Using BeautifulSoup to parse the HTML content of the Yelp page
soup = BeautifulSoup(r.text, 'html.parser')
# `r.text` gives the HTML content of the page.
# `BeautifulSoup(r.text, 'html.parser')` parses the HTML so that we can extract elements from it.

# Step 3: Creating a regular expression pattern to match the review class
regex = re.compile('.*comment__09f24__D0cxf y-css-1wfz87z.*')
# `re.compile()` is used to create a regular expression pattern.
# The pattern `'comment__09f24__D0cxf y-css-h9c2fl'` matches the class name used for review text on the Yelp page.
# The `.*` at the start and end allows for any characters before and after the specific part of the class name. This is useful when class names are dynamically generated.

# Step 4: Finding all `<p>` elements with the specified class using the regex pattern
results = soup.find_all('p', {'class': regex})
# `soup.find_all()` searches the parsed HTML (`soup`) for all `<p>` tags with the class that matches the regular expression.
# It returns a list of all elements that match the pattern.
# Here, we're specifically looking for `<p>` elements, as Yelp reviews are typically in paragraph tags.

# Step 5: Extracting the text of each review found and storing it in a list
reviews = [result.text for result in results]
# The list comprehension iterates over each `result` in `results` (each being a paragraph with the review text).
# For each review element, `.text` is used to get the text content (removing HTML tags).
# The reviews are then stored in the `reviews` list.

# The `reviews` list now contains all the extracted review texts from the Yelp page.

In [8]:
i=1
for review in reviews:
    print()
    print(f"Review {i} \n")
    print(textwrap.fill(review, width=150))
    i+=1
    print("--------------------------------------------------------------------------------------------------------------------------------------------------------")


Review 1 

"I've been there several times and I never get tired of it. The pizza is truly delicious, the service is 20/10, and the place is always clean and
pleasant. They should turn their policies into a franchise so that other restaurants can implement them."
--------------------------------------------------------------------------------------------------------------------------------------------------------

Review 2 

Came here for a business lunch and it did not disappoint! The pizza was good! Not oily like Pizza Hut or Dominos. Had the veggie one myself, but
everyone was satisfied with a number of the other menu items. Pradeep was very kind and accommodating. When asked for a small ranch, he gave an entire
tub worth for everyone to share easily. He made sure we had plates, parm & peppers, and to-go boxes after. The restaurant was clean and clean bathroom
as well. Things I'd improve: would make sure the remotes work so that you can play football games on the TV, especially on S

### Converting reviews to a dataframe

In [9]:
import pandas as pd
import numpy as np

In [10]:
## create a Dataframe with the reviews
df = pd.DataFrame(reviews, columns=['review'])
df.head(5)

,review
0,"""I've been there several times and I never get..."
1,Came here for a business lunch and it did not ...
2,More of a 3.5 than an actual 3.The order was p...
3,"Walking in, dude just staring. No greeting, no..."
4,"When it comes to chain store pizza , Mountain ..."


### Process the reviews for sentiment-analysis

In [11]:
def sentiment_score(review): ## Takes in one input of a review 
    tokens = tokenizer.encode(review, return_tensors='pt') ## Gets the token, just as we did before.
    result = model(tokens) ## Pass tokens to the model.
    return int(torch.argmax(result.logits))+1 ## selects the max prob adds 1 so that instead of [0-4] its [1-5]

In [12]:
def sentiment_rating():
  scorearr=[]
  for review in reviews: ## Takes 1 review and process it
    score = sentiment_score(review) ## Gets the score for that review
    scorearr.append(score) ## Appends the score in a array

  ## Overall Score, based on average
  return sum(scorearr)/len(scorearr) 




### Let's scrap the real rating, use the video linked and try to scrap the rating yourself!

In [25]:
## Extracting the real rating:
regex = re.compile('.*y-css-1jz061g.*')
results = soup.find_all('span', {'class': regex})
results=[result.text for result in results]

In [26]:
## As visible from the results there are multiple outputs, but we are interested only in the rating.
results

['4.2 ', 'Pizza, ', 'Chicken Wings', '11:00 AM - 1:00 AM (Next day)']

In [28]:
result = results[0]

In [31]:
print("Rating as predicted by our model: ", sentiment_rating())
print("Rating as scraped: ",result)

Rating as predicted by our model:  4.0
Rating as scraped:  4.2 


In [32]:
## The model isn't necessarily inaccurate; visit the website and read the scrapped reviews. Based on that make an inference on how close or far the model is from
## the desired output.